# Data Manipulation — Intermediate
## Reshaping & Combining Data

**Philosophy:** Assumes clean data and familiarity with the Pandas API. Each section covers a multi-step pattern — not individual method syntax, but how to chain tools together to solve a real reshaping problem.

---

## Decision Table

| If your problem is... | Go to |
| :--- | :--- |
| Data is wide and you need one row per observation | §1 — Wide vs Long |
| Need top-N per group, conditional aggregation, or filter-then-agg | §2 — Advanced GroupBy |
| Joining 3+ tables, row count explodes after merge, or need anti-join | §3 — Multi-table Pipelines |
| Need to change time frequency, align two time series, or forward-fill after resample | §4 — Time-Series Reshaping |
| Column contains JSON, dicts, or lists | §5 — Nested & List-like Data |
| Reading and combining many CSV or Excel files | §6 — Combining Multiple Files |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC — audit, dtypes, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` **(this note)** | Reshaping & combining — wide/long, groupby, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Pandas / Reference_NumPy` | Syntax lookups for individual methods |


> **Environment:** pandas 3.x / numpy 2.x. A small sample frame is defined below for runnable examples.

In [4]:
import pandas as pd, numpy as np

df = pd.DataFrame({
    'user_id':   [1, 1, 2, 2, 3],
    'region':    ['US','US','EU','EU','US'],
    'platform':  ['iOS','Android','iOS','iOS','Web'],
    'revenue':   [100, 150, 200, 120, 90],
    'event_date': pd.to_datetime(['2024-01-01','2024-02-01','2024-01-15','2024-03-01','2024-02-20']),
})
print(df)


   user_id region platform  revenue event_date
0        1     US      iOS      100 2024-01-01
1        1     US  Android      150 2024-02-01
2        2     EU      iOS      200 2024-01-15
3        2     EU      iOS      120 2024-03-01
4        3     US      Web       90 2024-02-20


---
## §1 — Wide vs Long Format

**Wide:** one row per entity, one column per time period or category. Easy to read, hard to aggregate.
**Long:** one row per observation, categories stored as values in a column. Required for most groupby/plotting/modeling operations.

In [ ]:
import pandas as pd
import numpy as np

# ── Wide → Long: melt ───────────────────────────────────────────────────────
# Wide format (common from Excel or pivot exports):
# user_id | Q1_revenue | Q2_revenue | Q3_revenue | Q4_revenue
wide = pd.DataFrame({
    'user_id':    [1, 2, 3],
    'Q1_revenue': [100, 200, 150],
    'Q2_revenue': [120, 210, 160],
    'Q3_revenue': [130, 190, 170],
})

long = wide.melt(
    id_vars    = ['user_id'],       # columns to keep as-is
    value_vars = ['Q1_revenue', 'Q2_revenue', 'Q3_revenue'],  # columns to unpivot
    var_name   = 'quarter',         # name for the new label column
    value_name = 'revenue'          # name for the new value column
)
# Result: user_id | quarter     | revenue
#         1       | Q1_revenue  | 100
#         1       | Q2_revenue  | 120  ...

# Clean the label column after melting
long['quarter'] = long['quarter'].str.replace('_revenue', '')

# ── Long → Wide: pivot_table ────────────────────────────────────────────────
# Reverse the operation — go back to one column per quarter
wide_again = long.pivot_table(
    index   = 'user_id',
    columns = 'quarter',
    values  = 'revenue',
    aggfunc = 'sum'
).reset_index()
wide_again.columns.name = None    # remove the 'quarter' name from the column axis

# ── When to use which format ────────────────────────────────────────────────
# Use LONG for:  groupby, plotting (most chart libraries), modeling
# Use WIDE for:  human-readable reports, correlation matrices, displaying to stakeholders

# ── Multiple value columns — pd.wide_to_long ────────────────────────────────
# wide_to_long expects columns shaped {stub}{sep}{suffix}, e.g. rev_Q1, cost_Q1.
# The `wide` frame above (Q1_revenue, ...) does NOT match that pattern and would
# return 0 rows. Build a frame whose names match the stubs:
wide2 = pd.DataFrame({
    'user_id': [1, 2],
    'rev_Q1': [100, 200], 'cost_Q1': [60, 110],
    'rev_Q2': [120, 210], 'cost_Q2': [70, 120],
})
df_long = pd.wide_to_long(
    wide2,
    stubnames = ['rev', 'cost'],   # column prefixes
    i         = 'user_id',
    j         = 'quarter',
    sep       = '_',
    suffix    = r'\w+'            # matches Q1, Q2, ...
).reset_index()
# df_long: user_id | quarter | rev | cost  (one row per user × quarter)

**Common mistakes:**
- Forgetting `reset_index()` after `pivot_table` — the index contains the row labels, not a regular column
- Leaving `columns.name` set after pivoting — causes confusing column axis label in displays
- Using `pivot` instead of `pivot_table` when there are duplicate index/column pairs — `pivot` raises an error; `pivot_table` aggregates them

**Verified — the corrected `wide_to_long`** (the §1 example reused a frame whose columns didn't match the stubs and silently returned 0 rows):

In [5]:
import pandas as pd
wide2 = pd.DataFrame({'user_id':[1,2],
                      'rev_Q1':[100,200], 'cost_Q1':[60,110],
                      'rev_Q2':[120,210], 'cost_Q2':[70,120]})
out = pd.wide_to_long(wide2, stubnames=['rev','cost'], i='user_id', j='quarter',
                      sep='_', suffix=r'\w+').reset_index()
print(out)


   user_id quarter  rev  cost
0        1      Q1  100    60
1        2      Q1  200   110
2        1      Q2  120    70
3        2      Q2  210   120


---
## §2 — Advanced GroupBy Patterns

Beyond basic `groupby().agg()` — patterns for filtering groups, selecting top-N within a group, and conditional aggregation. Syntax reference is in Pandas §6; this section focuses on the multi-step patterns.

In [ ]:
# ── Pattern 1: Top-N per group ───────────────────────────────────────────────
# Goal: top 3 products by revenue within each region
# Method A: rank + filter (preserves all columns, most flexible)
df['rev_rank'] = (
    df.groupby('region')['revenue']
      .rank(method='dense', ascending=False)
)
top3 = df[df['rev_rank'] <= 3].drop(columns='rev_rank')

# Method B: sort + groupby head (simpler but loses ranks)
top3 = (
    df.sort_values('revenue', ascending=False)
      .groupby('region', as_index=False)
      .head(3)
)

# ── Pattern 2: Filter groups by an aggregate condition ───────────────────────
# Goal: keep only regions where total revenue > 100,000
# groupby().filter() — predicate returns True/False per group
active_regions = df.groupby('region').filter(
    lambda g: g['revenue'].sum() > 100_000
)
# Result: same shape as df but entire groups removed if condition is False

# ── Pattern 3: Conditional aggregation ──────────────────────────────────────
# Goal: SUM(revenue WHERE platform = 'iOS') — SQL conditional agg equivalent
df['ios_rev']     = df['revenue'].where(df['platform'] == 'iOS', 0)
df['android_rev'] = df['revenue'].where(df['platform'] == 'Android', 0)

result = df.groupby('region', as_index=False).agg(
    total_rev   = ('revenue',     'sum'),
    ios_rev     = ('ios_rev',     'sum'),
    android_rev = ('android_rev', 'sum'),
    unique_users= ('user_id',     'nunique')
)
result['ios_share'] = result['ios_rev'] / result['total_rev']

# ── Pattern 4: Group-level features back on original rows ───────────────────
# Goal: add group mean, count, and pct-of-total back to each row
df['region_avg_rev'] = df.groupby('region')['revenue'].transform('mean')
df['region_count']   = df.groupby('region')['user_id'].transform('count')
df['pct_of_region']  = df['revenue'] / df.groupby('region')['revenue'].transform('sum')

# ── Pattern 5: Multi-level groupby ──────────────────────────────────────────
# Goal: revenue per region × platform × month
df['year_month'] = df['event_date'].dt.to_period('M').astype(str)
result = df.groupby(['region', 'platform', 'year_month'], as_index=False).agg(
    revenue = ('revenue', 'sum'),
    users   = ('user_id', 'nunique')
)

# ── Pattern 6: Custom aggregation function ───────────────────────────────────
# Goal: 90th percentile revenue per region
result = df.groupby('region', as_index=False).agg(
    p90_revenue = ('revenue', lambda x: x.quantile(0.9)),
    cv_revenue  = ('revenue', lambda x: x.std() / x.mean())  # coefficient of variation
)

**Pattern selection:**

| Goal | Tool |
| :--- | :--- |
| Top-N per group | `groupby().rank()` + filter, or `sort + groupby().head(N)` |
| Remove entire groups that fail a condition | `groupby().filter(lambda g: ...)` |
| Conditional sum/count within a group | `.where()` to zero-out + `agg()` |
| Group stats back on each row | `groupby().transform()` |
| Custom aggregation (percentile, CV) | `agg(col=('col', lambda x: ...))` |

**Common mistakes:**
- Using `agg` when `transform` is needed — `agg` collapses rows and loses the join-back to original data
- Using `groupby().filter()` when you only want to flag — filter removes rows permanently; use `transform` to add a boolean flag instead
- Forgetting `as_index=False` in groupby — group keys go into the index, not columns, breaking subsequent chaining

---
## §3 — Multi-table Pipelines

Single-merge operations are in the Pandas reference (§5). This section covers sequences of merges, diagnosing unexpected row count changes, and anti-join patterns.

In [ ]:
# ── Always check row counts at each merge step ───────────────────────────────
def merge_check(left, right, **kwargs):
    """Merge and print row count before/after to detect fan-out."""
    result = left.merge(right, **kwargs)
    print(f'{len(left)} → {len(result)} rows  ({len(result)-len(left):+d})')
    return result

# ── Pattern 1: Sequential 3-table join ──────────────────────────────────────
# users → orders → order_items
result = (
    users                                           # base: one row per user
    .merge(orders,      on='user_id',  how='left')  # left: keep all users
    .merge(order_items, on='order_id', how='left')  # left: keep all orders
)
# If users have multiple orders and orders have multiple items,
# row count grows: users × avg_orders × avg_items_per_order

# ── Diagnosing row count explosion ──────────────────────────────────────────
# Step 1: check if the join key is unique in the right table
print(orders.groupby('user_id').size().describe())
# If max > 1, the join will multiply rows — decide: aggregate right table first

# Step 2: aggregate the right table before joining (recommended for 1-to-many)
orders_agg = orders.groupby('user_id', as_index=False).agg(
    order_count  = ('order_id', 'count'),
    total_spend  = ('amount',   'sum'),
    last_order   = ('order_date','max')
)
result = users.merge(orders_agg, on='user_id', how='left')  # now 1-to-1

# ── Pattern 2: Anti-join — rows in A with no match in B ─────────────────────
# SQL: SELECT * FROM A WHERE A.id NOT IN (SELECT id FROM B)
merged = users.merge(
    churned_users[['user_id']],
    on='user_id', how='left', indicator=True
)
active_users = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')

# ── Pattern 3: Many-to-many join — handle explicitly ────────────────────────
# Tags table: one user can have many tags, one tag can belong to many users
# Direct merge → row explosion; resolve by checking what you actually need

# Option A: aggregate tags into a list per user first
tags_agg = tags.groupby('user_id')['tag'].agg(list).reset_index()
tags_agg.columns = ['user_id', 'tag_list']
result = users.merge(tags_agg, on='user_id', how='left')

# Option B: keep only one tag per user (e.g. most recent)
tags_dedup = tags.sort_values('created_at').drop_duplicates('user_id', keep='last')
result = users.merge(tags_dedup[['user_id','tag']], on='user_id', how='left')

# ── Pattern 4: Bridge table join (many-to-many via junction table) ───────────
# users ←→ user_roles ←→ roles
result = (
    users
    .merge(user_roles, on='user_id', how='inner')
    .merge(roles,      on='role_id', how='left')
)

**Row count change after merge — diagnostic guide:**

| Row count change | Likely cause | Fix |
| :--- | :--- | :--- |
| Count stays same | 1-to-1 match ✅ | — |
| Count increases | Right table has multiple rows per key | Aggregate right table first |
| Count decreases | Inner join dropped non-matching rows | Switch to `how='left'` if needed |
| Count drops to near zero | Join key dtype mismatch (int vs str) | Cast keys to same type |

**Common mistakes:**
- Joining without checking key uniqueness — row explosion is silent and hard to detect later
- Joining on columns with different dtypes — pandas produces zero matches with no error; always verify `df['id'].dtype`
- Using `how='inner'` when you need all rows from the left table — rows drop silently

**Two §3 additions: `merge(validate=)` and `combine_first`.**

The `merge_check` helper above prints row counts to *detect* fan-out after the fact; `merge(..., validate=...)` is the built-in *guard* that raises a `MergeError` the moment a join would violate the cardinality you expect — make it the first line of defence. `combine_first` coalesces two frames, filling gaps in the primary from a backup, aligned on index.

In [6]:
import pandas as pd
users  = pd.DataFrame({'user_id':[1,2,3], 'name':['a','b','c']})
orders = pd.DataFrame({'user_id':[1,1,2], 'amt':[10,20,30]})   # user 1 appears twice

try:
    users.merge(orders, on='user_id', how='left', validate='one_to_one')
except Exception as e:
    print('validate="one_to_one" ->', type(e).__name__, ':', str(e).splitlines()[0])
print('validate="one_to_many" -> rows:',
      len(users.merge(orders, on='user_id', how='left', validate='one_to_many')))

# combine_first — fill NaNs in primary from backup, index-aligned
primary = pd.Series([1, None, 3])
backup  = pd.Series([9, 2,    9])
print('combine_first        :', primary.combine_first(backup).tolist())


validate="one_to_one" -> MergeError : Merge keys are not unique in right dataset; not a one-to-one merge
validate="one_to_many" -> rows: 4
combine_first        : [1.0, 2.0, 3.0]


---
## §4 — Time-Series Reshaping

Covers changing temporal granularity, filling gaps in time series, and aligning two series that run on different frequencies.

In [ ]:
# Setup: daily transaction data
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()

# ── Pattern 1: Downsample — daily → weekly / monthly ────────────────────────
monthly = df['revenue'].resample('ME').sum()      # month-end frequency, sum
weekly  = df['revenue'].resample('W').mean()      # weekly, mean
# Common freq aliases: 'D'=daily, 'W'=weekly, 'ME'=month-end,
#                      'QE'=quarter-end, 'YE'=year-end, 'h'=hourly

# Multiple columns at different aggregations
monthly = df.resample('ME').agg({
    'revenue':  'sum',
    'users':    'nunique',
    'sessions': 'count'
})

# ── Pattern 2: Forward-fill gaps after resample ──────────────────────────────
# Problem: resampling creates NaN for periods with no data
# Use case: subscription status — carry last known status forward
daily_status = df['subscription_status'].resample('D').last()  # last known per day
daily_status = daily_status.ffill()  # fill gaps with previous known status

# ── Pattern 3: Upsample — monthly → daily ───────────────────────────────────
# Go from lower to higher frequency (creates NaN for new dates)
daily = monthly_series.resample('D').asfreq()   # insert daily rows, fill with NaN
daily = monthly_series.resample('D').ffill()    # forward fill from monthly value
daily = monthly_series.resample('D').interpolate('linear')  # linear interpolation

# ── Pattern 4: Align two time series with different frequencies ──────────────
# Example: daily sales + monthly macro indicator
daily_sales    = df_daily.set_index('date')['revenue']
monthly_macro  = df_macro.set_index('date')['gdp_growth']

# Reindex monthly macro to daily frequency, then forward fill
macro_daily = monthly_macro.reindex(daily_sales.index).ffill()
aligned = pd.DataFrame({'revenue': daily_sales, 'gdp_growth': macro_daily})

# ── Pattern 5: Create a complete date spine ──────────────────────────────────
# Problem: some dates are missing entirely from the data
# Solution: build a full date range and left join
date_spine = pd.date_range(
    start=df['date'].min(),
    end=df['date'].max(),
    freq='D'
)
full_dates = pd.DataFrame({'date': date_spine})
complete = full_dates.merge(df, on='date', how='left')
complete['revenue'] = complete['revenue'].fillna(0)  # missing days = 0 revenue

**Common mistakes:**
- Forward-filling before checking sort order — `ffill()` fills in index order; always sort the index first
- Using `resample()` on a DataFrame without a DatetimeIndex — set the date column as index first with `set_index()`
- Downsampling with `mean()` when `sum()` is correct — e.g. daily revenue should be summed, not averaged, when going monthly
- Creating a date spine with `pd.date_range` but the DataFrame dates have time components — normalize first with `df['date'].dt.normalize()`

---
## §5 — Nested & List-like Data

Common when working with API responses, JSON exports, or data where one cell contains a list or dict. Pandas has dedicated tools for flattening these structures.

In [ ]:
# ── Pattern 1: Flatten JSON / dict column ────────────────────────────────────
# Data: df['metadata'] contains dicts like {'source': 'organic', 'campaign': 'summer'}
meta_df = pd.json_normalize(df['metadata'])      # expands each key to a column
df = pd.concat([df.drop(columns='metadata'), meta_df], axis=1)

# Nested JSON (multiple levels deep)
# Data: {'user': {'id': 1, 'profile': {'age': 30, 'city': 'NYC'}}}
flat = pd.json_normalize(
    records,
    sep='_',           # separator for nested keys: user_profile_age
    max_level=2        # limit depth
)

# ── Pattern 2: Explode a list column ─────────────────────────────────────────
# Data: df['tags'] contains ['python', 'pandas', 'sql']
# Goal: one row per tag
df_exploded = df.explode('tags').reset_index(drop=True)
# Before: user_id=1, tags=['python','pandas'] → 1 row
# After:  user_id=1, tags='python' AND user_id=1, tags='pandas' → 2 rows

# Explode multiple list columns of the same length simultaneously
df_exploded = df.explode(['tags', 'scores']).reset_index(drop=True)

# ── Pattern 3: Parse stringified lists ──────────────────────────────────────
# Data: df['tags'] contains the string "['python', 'pandas']"
import ast
df['tags'] = df['tags'].apply(ast.literal_eval)   # safe string → Python list
df_exploded = df.explode('tags')

# ── Pattern 4: Expand a list column into separate columns ───────────────────
# Data: df['scores'] contains [84, 91, 76] (fixed length)
scores_df = pd.DataFrame(
    df['scores'].tolist(),
    columns=['score_1', 'score_2', 'score_3']
)
df = pd.concat([df.drop(columns='scores'), scores_df], axis=1)

# ── Pattern 5: Flatten nested JSON from an API response ─────────────────────
import json
# API response: list of dicts with nested structure
records = json.loads(api_response_text)
df = pd.json_normalize(
    records,
    record_path = ['orders'],          # the list to expand
    meta        = ['user_id', 'name'], # top-level fields to keep on each row
    sep         = '_'
)

**Common mistakes:**
- Using `.explode()` without `reset_index(drop=True)` — original index repeats for each exploded row, creating a non-unique index
- Calling `pd.json_normalize(df['col'])` when the column has NaN values — raises `AttributeError`; fill with `{}` first
- Using `eval()` to parse stringified lists instead of `ast.literal_eval` — `eval()` is a security risk; `literal_eval` is safe and rejects arbitrary code

---
## §6 — Combining Multiple Files

Common in real-world data pipelines: monthly data exports, partitioned data dumps, or multi-sheet Excel files. The challenge is schema consistency across files.

In [ ]:
import glob
from pathlib import Path

# ── Pattern 1: Read and concat many CSVs ────────────────────────────────────
# Simple — same schema across all files
files = glob.glob('data/monthly/*.csv')          # or Path('data/monthly').glob('*.csv')
df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True                            # re-index to avoid duplicate index values
)

# ── Pattern 2: Add a source column (track which file each row came from) ─────
dfs = []
for f in files:
    chunk = pd.read_csv(f)
    chunk['source_file'] = Path(f).stem          # e.g. '2024-01'
    dfs.append(chunk)
df = pd.concat(dfs, ignore_index=True)

# ── Pattern 3: Validate schema before concat ─────────────────────────────────
# Problem: files from different periods may have different columns or dtypes
dfs = [pd.read_csv(f) for f in files]

# Check column consistency
expected_cols = set(dfs[0].columns)
for i, d in enumerate(dfs[1:], 1):
    missing  = expected_cols - set(d.columns)
    extra    = set(d.columns) - expected_cols
    if missing or extra:
        print(f'File {i}: missing={missing}, extra={extra}')

# Safe concat — aligns on column names, fills missing with NaN
df = pd.concat(dfs, ignore_index=True, join='outer')  # outer keeps all columns

# ── Pattern 4: Read all sheets from an Excel file ────────────────────────────
xl = pd.read_excel('report.xlsx', sheet_name=None)   # returns dict: {sheet_name: df}
df = pd.concat(xl.values(), ignore_index=True)       # concat all sheets

# Or concat with sheet name as a column
dfs = []
for sheet_name, sheet_df in xl.items():
    sheet_df['sheet'] = sheet_name
    dfs.append(sheet_df)
df = pd.concat(dfs, ignore_index=True)

# ── Pattern 5: Large files — read selectively ────────────────────────────────
# Only load the columns you need to avoid memory issues
needed_cols = ['user_id', 'date', 'revenue']
df = pd.concat(
    [pd.read_csv(f, usecols=needed_cols) for f in files],
    ignore_index=True
)

**Common mistakes:**
- `pd.concat` without `ignore_index=True` — original row indices repeat across files, creating duplicate index values that break `.loc` lookups
- Assuming all files have the same schema — always validate before concat, especially with data from different time periods
- Loading all columns from all files when only a few are needed — use `usecols=` to limit memory usage
- Using `pd.concat([df1, df2])` with mismatched dtypes (e.g. `int` in one file, `float` in another) — pandas upcasts silently; verify with `df.dtypes` after concat

---
## Decision Guide

```
Need to change data shape?
├── Wide → Long (many columns → one value column)  → .melt(id_vars, value_vars)          (§1)
├── Long → Wide (categories → separate columns)    → .pivot_table(index, columns, values) (§1)
└── Multi-value wide → long                        → pd.wide_to_long()                   (§1)

Need advanced groupby?
├── Top-N per group                                → groupby().rank() + filter            (§2)
├── Remove groups failing a condition              → groupby().filter(lambda g: ...)      (§2)
├── Conditional sum within group (SUM CASE WHEN)   → .where() + groupby().agg()          (§2)
├── Group stats back on each row                   → groupby().transform()                (§2)
└── Custom aggregation (percentile, ratio)         → agg(col=('col', lambda x: ...))     (§2)

Combining multiple tables?
├── Row count grows after merge                    → aggregate right table first          (§3)
├── Rows in A with no match in B                   → merge(how='left', indicator=True)    (§3)
├── Many-to-many relationship                      → aggregate one side or use bridge     (§3)
└── Key dtype mismatch (rows drop to zero)         → cast both keys to same type          (§3)

Working with time series?
├── Change frequency (daily → monthly)             → .resample('ME').agg()               (§4)
├── Fill gaps in time series                       → build date spine + left merge        (§4)
├── Carry last known value forward                 → .resample().last().ffill()           (§4)
└── Align different frequencies                    → .reindex(other.index).ffill()        (§4)

Column contains dicts / JSON?
└──                                                → pd.json_normalize()                  (§5)

Column contains lists?
├── One row per list element                       → .explode()                           (§5)
├── Fixed-length list → separate columns           → pd.DataFrame(df['col'].tolist())     (§5)
└── Stringified list ("['a','b']")                → ast.literal_eval + .explode()        (§5)

Reading many files?
├── Same schema, just concat                       → glob + pd.concat(ignore_index=True)  (§6)
├── Track source file                              → add 'source_file' column per chunk   (§6)
├── Schema may differ across files                 → validate columns first, join='outer' (§6)
└── Multiple Excel sheets                          → read_excel(sheet_name=None)          (§6)
```